In [0]:
# Bronze Layer - Raw Ingestion
# Reads CSV files from landing Volume, applies schema, adds audit columns,
# writes to Delta Lake Bronze tables in finpulse_dev.bronze schema
# Incremental: tracks processed files to avoid reprocessing on each run

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, current_timestamp
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, TimestampType, DateType
)
from delta.tables import DeltaTable
from datetime import datetime

spark = SparkSession.builder.getOrCreate()

VOLUME_PATH   = "/Volumes/finpulse_dev/raw_landing/source_files"
CATALOG       = "finpulse_dev"
BRONZE_SCHEMA = "bronze"

# Create bronze schema if not exists
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{BRONZE_SCHEMA}")
print(f"Bronze schema ready: {CATALOG}.{BRONZE_SCHEMA}")

branches_schema = StructType([
    StructField("branch_id",   StringType(), True),
    StructField("branch_name", StringType(), True),
    StructField("city",        StringType(), True),
    StructField("state",       StringType(), True),
    StructField("branch_type", StringType(), True)
])

customers_schema = StructType([
    StructField("customer_id",            StringType(), True),
    StructField("first_name",             StringType(), True),
    StructField("last_name",              StringType(), True),
    StructField("date_of_birth",          StringType(), True),
    StructField("email",                  StringType(), True),
    StructField("phone",                  StringType(), True),
    StructField("address_line",           StringType(), True),
    StructField("city",                   StringType(), True),
    StructField("state",                  StringType(), True),
    StructField("pincode",                StringType(), True),
    StructField("kyc_status",             StringType(), True),
    StructField("customer_since_date",    StringType(), True),
    StructField("last_updated_timestamp", StringType(), True)
])

accounts_schema = StructType([
    StructField("account_id",             StringType(), True),
    StructField("customer_id",            StringType(), True),
    StructField("account_type",           StringType(), True),
    StructField("branch_id",              StringType(), True),
    StructField("open_date",              StringType(), True),
    StructField("status",                 StringType(), True),
    StructField("currency",               StringType(), True),
    StructField("last_updated_timestamp", StringType(), True)
])

cards_schema = StructType([
    StructField("card_id",      StringType(), True),
    StructField("account_id",   StringType(), True),
    StructField("card_type",    StringType(), True),
    StructField("card_network", StringType(), True),
    StructField("issue_date",   StringType(), True),
    StructField("expiry_date",  StringType(), True),
    StructField("status",       StringType(), True)
])

transactions_schema = StructType([
    StructField("transaction_id",        StringType(), True),
    StructField("account_id",            StringType(), True),
    StructField("transaction_timestamp", StringType(), True),
    StructField("amount",                StringType(), True),
    StructField("currency",              StringType(), True),
    StructField("transaction_type",      StringType(), True),
    StructField("channel",               StringType(), True),
    StructField("merchant_name",         StringType(), True),
    StructField("status",                StringType(), True),
    StructField("ingestion_timestamp",   StringType(), True)
])

print("All schemas defined successfully")

In [0]:
# -------------------------------------------------------------------
# Bronze ingestion function
# Reusable across all entities - same pattern every time:
# 1. Read CSV with explicit schema
# 2. Add audit columns
# 3. Write to Delta table (append mode)
# -------------------------------------------------------------------

def ingest_to_bronze(
    entity_name,
    source_path,
    schema,
    partition_col=None
):
    print(f"\n{'='*50}")
    print(f"Ingesting: {entity_name}")
    print(f"Source   : {source_path}")

    # Read CSV with explicit schema
    # PERMISSIVE mode: malformed records get nulls instead of crashing pipeline
    df = spark.read \
        .option("header", True) \
        .option("mode", "PERMISSIVE") \
        .schema(schema) \
        .csv(source_path)

    raw_count = df.count()
    print(f"Raw records read : {raw_count}")

    # Add audit columns
    # _ingested_at : when our pipeline processed this record
    # _source_file : exact file path this record came from (UC-compatible)
    # _load_date   : partition-friendly date for querying by load date
    df_with_audit = df \
        .withColumn("_ingested_at", current_timestamp()) \
        .withColumn("_source_file", col("_metadata.file_path")) \
        .withColumn("_load_date",   lit(datetime.now().strftime("%Y-%m-%d")))

    # Write to Delta table
    target_table = f"{CATALOG}.{BRONZE_SCHEMA}.{entity_name}"

    if partition_col:
        df_with_audit.write \
            .format("delta") \
            .mode("append") \
            .partitionBy(partition_col) \
            .saveAsTable(target_table)
    else:
        df_with_audit.write \
            .format("delta") \
            .mode("append") \
            .saveAsTable(target_table)

    # Verify write
    written_count = spark.table(target_table).count()
    print(f"Records in Delta table : {written_count}")
    print(f"Target table           : {target_table}")
    print(f"Status                 : SUCCESS")

    return written_count

print("Bronze ingestion function defined successfully")

In [0]:
# -------------------------------------------------------------------
# Run Bronze ingestion for all entities
# Order matters: reference data first (branches), then dimensions
# (customers, accounts, cards), then facts (transactions)
# -------------------------------------------------------------------

print("Starting Bronze ingestion pipeline...")
print(f"Run timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

# Order: reference data → dimensions → facts
# 1. Branches
ingest_to_bronze(
    entity_name  = "branches",
    source_path  = f"{VOLUME_PATH}/branches/",
    schema       = branches_schema,
    partition_col= None
)

# 2. Customers
ingest_to_bronze(
    entity_name  = "customers",
    source_path  = f"{VOLUME_PATH}/customers/",
    schema       = customers_schema,
    partition_col= "_load_date"
)

# 3. Accounts
ingest_to_bronze(
    entity_name  = "accounts",
    source_path  = f"{VOLUME_PATH}/accounts/",
    schema       = accounts_schema,
    partition_col= "_load_date"
)

# 4. Cards
ingest_to_bronze(
    entity_name  = "cards",
    source_path  = f"{VOLUME_PATH}/cards/",
    schema       = cards_schema,
    partition_col= "_load_date"
)

# 5. Transactions
ingest_to_bronze(
    entity_name  = "transactions",
    source_path  = f"{VOLUME_PATH}/transactions/",
    schema       = transactions_schema,
    partition_col= "_load_date"
)

print(f"\n{'='*50}")
print("Bronze ingestion pipeline complete!")

In [0]:
# Spot check audit columns on customers
display(
    spark.table("finpulse_dev.bronze.customers")
    .select(
        "customer_id",
        "first_name",
        "kyc_status",
        "_ingested_at",
        "_source_file",
        "_load_date"
    )
    .limit(5)
)

In [0]:
# Delta table history - every operation ever performed on this table
# In production this is invaluable for debugging ("who wrote what and when")
display(
    spark.sql("DESCRIBE HISTORY finpulse_dev.bronze.customers")
)

In [0]:
# Bronze layer health check
# Run this after every Bronze pipeline execution to confirm all tables
# are populated and audit columns are present

bronze_tables = ["branches", "customers", "accounts", "cards", "transactions"]

print("=" * 55)
print(f"{'Table':<20} {'Row Count':>12} {'Load Dates':>20}")
print("=" * 55)

for table in bronze_tables:
    df = spark.table(f"finpulse_dev.bronze.{table}")
    row_count = df.count()
    load_dates = [
        row["_load_date"] 
        for row in df.select("_load_date").distinct().collect()
    ]
    load_dates_str = ", ".join(sorted(load_dates))
    print(f"{table:<20} {row_count:>12,} {load_dates_str:>20}")

print("=" * 55)
print("Bronze health check complete")